#### This section demonstrates the creation of the GOLD layer **dim_customer table** using data sourced from the SILVER layer.

#### The other dimension tables : **dim_product, dim_date, and dim_salesregion**, follow the same implementation pattern, with adjustments made only to table names and data paths.

In [0]:
from pyspark.sql.functions import *

### Creating a widget for Incremental Run

In [0]:
dbutils.widgets.text("Incremental FLAG", "0")

### Checking the Incremental Run

In [0]:
inc_flg = dbutils.widgets.get("Incremental FLAG")

### Establishing the connection with ADLS

In [0]:
spark.conf.set("fs.azure.account.key.sa1storage.dfs.core.windows.net", "G1RfRXHyK97KaTGSJXAD1RqeXkcX0WRYxtUi4g/pUZx7GxFi3MWAEvU7uU9yh7I27Syh/IUORuea+AStsT8Lzw==")

### Checking the SILVER connection

In [0]:
%sql

SELECT *
FROM parquet.`abfss://silver@sa1storage.dfs.core.windows.net/cleanseddata`
LIMIT 20

OrderID,OrderDate,CustomerID,CustomerName,CustomerEmail,Country,ProductID,ProductCategory,ProductName,Quantity,UnitPrice,TotalPrice,SalesRegion
38,2024-04-10,1696,George Wright,ibarnes@davis.com,Guatemala,579,Sports,Phone,5,583.72,2918.60,Central
433,2025-02-06,1121,Rachael Paul,rachael01@richmond.com,Bahrain,536,Furniture,Laptop,8,178.08,1424.64,Central
464,2024-09-24,1939,Matthew Lane,perryjoseph@gmail.com,Czech Republic,538,Clothing,Laptop,3,891.24,2673.72,West
618,2025-02-09,1354,Douglas Park,erowland@davis-doyle.info,Bulgaria,573,Furniture,Shoes,5,182.88,914.40,Central
632,2023-10-28,1187,Kelly Fisher,iparker@yahoo.com,Panama,539,Furniture,Shoes,9,782.59,7043.31,West
885,2023-10-09,1990,Gary Ray,christopherwilson@gmail.com,Saint Martin,552,Clothing,Laptop,1,842.54,842.54,East
969,2023-08-22,1249,Christine Curtis,thomas75@yahoo.com,Northern Mariana Islands,598,Electronics,Perfume,4,145.78,583.12,South
6,2025-06-12,1525,Elizabeth Cunningham,karen47@lee-cox.com,Azerbaijan,537,Clothing,Shoes,5,978.02,4890.10,West
112,2023-10-02,1442,Michelle Craig,christianguerrero@hotmail.com,Nepal,553,Beauty,Table,9,84.37,759.33,North
311,2024-11-21,1196,Rhonda Hardy,alexis46@hotmail.com,Mongolia,545,Electronics,Bicycle,4,13.15,52.60,West


### Working on Customer dimension

In [0]:
%sql

SELECT DISTINCT(CustomerID) as Cust_ID, CustomerName as Cust_Name, CustomerEmail as Email, Country
FROM parquet.`abfss://silver@sa1storage.dfs.core.windows.net/cleanseddata`
LIMIT 20

Cust_ID,Cust_Name,Email,Country
1935,Beth Hernandez,aliciamarks@gmail.com,India
1181,James Hurley,skinnerrussell@lowery-chaney.com,Iraq
1248,Nicholas Jensen,garciaangela@levine.net,New Zealand
1410,Amanda Lopez,donaldmendoza@richard-anderson.com,Namibia
1989,Andre Lin,jacobsmith@skinner-chapman.com,Nepal
1349,Alex Stone,jordantaylor@johnson.com,Equatorial Guinea
1669,Diana Eaton,kaylathompson@miranda.com,Georgia
1944,Kyle Gray,jesus71@ortiz.com,Myanmar
1257,Kurt Roberts,hansenanthony@vargas.net,Trinidad and Tobago
1840,Aaron Hall,bethwilson@miles.com,Turkey


#### SILVER to GOLD

#### S2G(1) - Storing the Customer data in DataFrame

In [0]:
df_src_cust = spark.sql(
 """
 SELECT DISTINCT CustomerID as Cust_ID, CustomerName as Cust_Name, CustomerEmail as Email, Country
 FROM parquet.`abfss://silver@sa1storage.dfs.core.windows.net/cleanseddata`""" 

)

df_src_cust.limit(20).display()

Cust_ID,Cust_Name,Email,Country
1935,Beth Hernandez,aliciamarks@gmail.com,India
1181,James Hurley,skinnerrussell@lowery-chaney.com,Iraq
1248,Nicholas Jensen,garciaangela@levine.net,New Zealand
1410,Amanda Lopez,donaldmendoza@richard-anderson.com,Namibia
1989,Andre Lin,jacobsmith@skinner-chapman.com,Nepal
1349,Alex Stone,jordantaylor@johnson.com,Equatorial Guinea
1669,Diana Eaton,kaylathompson@miranda.com,Georgia
1944,Kyle Gray,jesus71@ortiz.com,Myanmar
1257,Kurt Roberts,hansenanthony@vargas.net,Trinidad and Tobago
1840,Aaron Hall,bethwilson@miles.com,Turkey


#### S2G(2) - Getting existing GOLD Layer data

In [0]:
from delta.tables import DeltaTable

path = 'abfss://gold@sa1storage.dfs.core.windows.net/dim_customer'

if DeltaTable.isDeltaTable(spark, path):

  df_sink_cust = spark.sql(
    """
    SELECT Dim_Cust_Key, Cust_ID, Cust_Name, Email, Country
    FROM delta.`abfss://gold@sa1storage.dfs.core.windows.net/dim_customer`
    """
  )
  print('The TABLE already exists')

else:
  df_sink_cust = spark.sql(
    """
    SELECT 1 as Dim_Cust_Key, CustomerID as Cust_ID, CustomerName as Cust_Name, CustomerEmail as Email, Country
    FROM parquet.`abfss://silver@sa1storage.dfs.core.windows.net/cleanseddata`
    WHERE 1=2       
    """
  )
  # doing the above in WHERE so that only column names are returned
  
# OR
# df_cust_sink = spark.createDataFrame([], schema = "Dim_Cust_Key int, Cust_ID int, Cust_Name string, Email string, Country string")

df_sink_cust.display()


Dim_Cust_Key,Cust_ID,Cust_Name,Email,Country


#### S2G(3) - Doing the LEFT Join between 1 & 2

In [0]:
df = df_src_cust.join(df_sink_cust, df_src_cust['Cust_ID'] == df_sink_cust['Cust_ID'], "left").select(df_src_cust['Cust_ID'], df_src_cust['Cust_Name'], df_src_cust['Email'], df_src_cust['Country'], df_sink_cust['Dim_Cust_Key'])

df.limit(20).display()

Cust_ID,Cust_Name,Email,Country,Dim_Cust_Key
1935,Beth Hernandez,aliciamarks@gmail.com,India,null
1181,James Hurley,skinnerrussell@lowery-chaney.com,Iraq,null
1248,Nicholas Jensen,garciaangela@levine.net,New Zealand,null
1410,Amanda Lopez,donaldmendoza@richard-anderson.com,Namibia,null
1989,Andre Lin,jacobsmith@skinner-chapman.com,Nepal,null
1349,Alex Stone,jordantaylor@johnson.com,Equatorial Guinea,null
1669,Diana Eaton,kaylathompson@miranda.com,Georgia,null
1944,Kyle Gray,jesus71@ortiz.com,Myanmar,null
1257,Kurt Roberts,hansenanthony@vargas.net,Trinidad and Tobago,null
1840,Aaron Hall,bethwilson@miles.com,Turkey,null


#### S2G(4) - Separating the OLD and NEW record

In [0]:
df_old = df.filter(df['Dim_Cust_Key'].isNotNull())
df_old.limit(20).display()

# we'll get Nothing, since this is the first run and we don't have any old records

Cust_ID,Cust_Name,Email,Country,Dim_Cust_Key


In [0]:
df_new = df.filter(df['Dim_Cust_Key'].isNull())
df_new.limit(20).display()

Cust_ID,Cust_Name,Email,Country,Dim_Cust_Key
1935,Beth Hernandez,aliciamarks@gmail.com,India,null
1181,James Hurley,skinnerrussell@lowery-chaney.com,Iraq,null
1248,Nicholas Jensen,garciaangela@levine.net,New Zealand,null
1410,Amanda Lopez,donaldmendoza@richard-anderson.com,Namibia,null
1989,Andre Lin,jacobsmith@skinner-chapman.com,Nepal,null
1349,Alex Stone,jordantaylor@johnson.com,Equatorial Guinea,null
1669,Diana Eaton,kaylathompson@miranda.com,Georgia,null
1944,Kyle Gray,jesus71@ortiz.com,Myanmar,null
1257,Kurt Roberts,hansenanthony@vargas.net,Trinidad and Tobago,null
1840,Aaron Hall,bethwilson@miles.com,Turkey,null


#### S2G(5) - Assign the Dim_Cust_Key to the NEW Records

In [0]:
# Getting the max Dim_Cust_Key from the OLD Records

if inc_flg == '0':
  max_value = 1
else:
  max_value = df_old.agg(max("Dim_Cust_Key")).collect()[0][0]      #max_value = df_old.filter(max(df_old('Dim_Cust_Key'))).collect()[0][0]

max_value

1

In [0]:
# Creating surrogate_keys

df_new = df_new.withColumn('Dim_Cust_Key', max_value + monotonically_increasing_id())

df_new.limit(20).display()


Cust_ID,Cust_Name,Email,Country,Dim_Cust_Key
1935,Beth Hernandez,aliciamarks@gmail.com,India,1
1181,James Hurley,skinnerrussell@lowery-chaney.com,Iraq,2
1248,Nicholas Jensen,garciaangela@levine.net,New Zealand,3
1410,Amanda Lopez,donaldmendoza@richard-anderson.com,Namibia,4
1989,Andre Lin,jacobsmith@skinner-chapman.com,Nepal,5
1349,Alex Stone,jordantaylor@johnson.com,Equatorial Guinea,6
1669,Diana Eaton,kaylathompson@miranda.com,Georgia,7
1944,Kyle Gray,jesus71@ortiz.com,Myanmar,8
1257,Kurt Roberts,hansenanthony@vargas.net,Trinidad and Tobago,9
1840,Aaron Hall,bethwilson@miles.com,Turkey,10


In [0]:
# UNION the OLD Records and NEW Records having surrogate keys together

df = df_old.union(df_new)

df.limit(20).display()


Cust_ID,Cust_Name,Email,Country,Dim_Cust_Key
1935,Beth Hernandez,aliciamarks@gmail.com,India,1
1181,James Hurley,skinnerrussell@lowery-chaney.com,Iraq,2
1248,Nicholas Jensen,garciaangela@levine.net,New Zealand,3
1410,Amanda Lopez,donaldmendoza@richard-anderson.com,Namibia,4
1989,Andre Lin,jacobsmith@skinner-chapman.com,Nepal,5
1349,Alex Stone,jordantaylor@johnson.com,Equatorial Guinea,6
1669,Diana Eaton,kaylathompson@miranda.com,Georgia,7
1944,Kyle Gray,jesus71@ortiz.com,Myanmar,8
1257,Kurt Roberts,hansenanthony@vargas.net,Trinidad and Tobago,9
1840,Aaron Hall,bethwilson@miles.com,Turkey,10


#### S2D(6) - UPSERT (Update + Insert)

##### Till now, we have worked on Parquet Table. Now, to use UPSERT, we'll work on Delta Table

##### Delta Table, Parquet Table ke upar bani hui file hai which follows ACID Transactions

###### Also, note that, we'll check our table in the specified path, not in the Databricks Catalog, bcoz we are not using that

In [0]:
from delta.tables import DeltaTable

table_name = 'gold.Dim_Customer'

path = 'abfss://gold@sa1storage.dfs.core.windows.net/Dim_Customer'

if DeltaTable.isDeltaTable(spark, path):      # prev : if spark.catalog.tableExists(table_name): (not needed anymore)
  deltaTable = DeltaTable.forPath(spark, path)
  deltaTable.alias('target').merge(df.alias('source'), 'target.Dim_Cust_Key = source.Dim_Cust_Key').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
  print('The TABLE already exits')

else:
  df.write.format('delta').mode('overwrite').option('mergeschema', 'true').save(path)
  print('The TABLE is created')

The TABLE already exits


In [0]:
%sql
SELECT *
FROM delta.`abfss://gold@sa1storage.dfs.core.windows.net/Dim_Customer`
LIMIT 20

Cust_ID,Cust_Name,Email,Country,Dim_Cust_Key
1935,Beth Hernandez,aliciamarks@gmail.com,India,1
1181,James Hurley,skinnerrussell@lowery-chaney.com,Iraq,2
1248,Nicholas Jensen,garciaangela@levine.net,New Zealand,3
1410,Amanda Lopez,donaldmendoza@richard-anderson.com,Namibia,4
1989,Andre Lin,jacobsmith@skinner-chapman.com,Nepal,5
1349,Alex Stone,jordantaylor@johnson.com,Equatorial Guinea,6
1669,Diana Eaton,kaylathompson@miranda.com,Georgia,7
1944,Kyle Gray,jesus71@ortiz.com,Myanmar,8
1257,Kurt Roberts,hansenanthony@vargas.net,Trinidad and Tobago,9
1840,Aaron Hall,bethwilson@miles.com,Turkey,10
